<a href="https://colab.research.google.com/github/Sebastian771-177/Watchtower/blob/main/Watchtower.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install anthropic requests --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 10.8 MB/s eta 0:00:00


In [3]:
import anthropic
import requests
import json
import re
import time
from datetime import datetime, UTC
from IPython.display import display, HTML, clear_output
from google.colab import userdata

In [4]:
try:
    ANTHROPIC_KEY = userdata.get("ANTHROPIC_API_KEY")
    VT_KEY = userdata.get("VIRUSTOTAL_API_KEY")
    ABUSEIPDB_KEY = userdata.get("ABUSEIPDB_API_KEY")
except Exception:
    import os
    ANTHROPIC_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
    VT_KEY = os.environ.get("VIRUSTOTAL_API_KEY", "")
    ABUSEIPDB_KEY = os.environ.get("ABUSEIPDB_API_KEY", "")

client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

In [5]:
SESSION = {
    "started": datetime.now(UTC).strftime("%Y-%m-%d %H:%M UTC"),
    "centinela_reports": [],
    "sentinel_reports": [],
    "total_analyzed": 0,
    "severity_counts": {"CRITICAL": 0, "HIGH": 0, "MEDIUM": 0, "LOW": 0},
    "verdict_counts": {"MALICIOUS": 0, "SUSPICIOUS": 0, "CLEAN": 0, "UNKNOWN": 0},
}

In [6]:
CENTINELA_FEEDS = [
    {
        "id": "MX-2025-001",
        "source": "Foro Clandestino MX",
        "timestamp": "2025-04-25 02:14 UTC",
        "region": "México",
        "raw_text": (
            "Se vende acceso RDP a empresa financiera CDMX. Panel admin, "
            "500 equipos, sin AV activo. Precio: $800 USD crypto. "
            "Telegram: @sombra_red. Entrega inmediata. Solo BTC o XMR."
        ),
    },
    {
        "id": "AR-2025-002",
        "source": "Foro Hacktivist AR",
        "timestamp": "2025-04-24 23:31 UTC",
        "region": "Argentina",
        "raw_text": (
            "Operación en curso contra Ministerio de Economía Argentina. "
            "Tenemos acceso a base de datos de contribuyentes. Exigimos transparencia "
            "fiscal o publicamos 2.3 millones de registros el viernes. #OpArgentina"
        ),
    },
    {
        "id": "CL-2025-003",
        "source": "Red Oscura CL",
        "timestamp": "2025-04-24 19:45 UTC",
        "region": "Chile",
        "raw_text": (
            "Nuevo grupo RaaS: 'Cóndor Negro'. Afiliados reciben 75% del rescate. "
            "Ya comprometimos 3 clínicas en Santiago. Buscamos afiliados con acceso "
            "a redes corporativas. Contacto solo por I2P."
        ),
    },
]

CENTINELA_MOCK = [
    {
        "severity": "CRITICAL",
        "threat_type": "Initial Access Broker",
        "summary": "A threat actor is selling RDP access to a financial institution in Mexico City with admin-level privileges across 500 endpoints. The absence of active antivirus significantly lowers the barrier for follow-on ransomware or data exfiltration.",
        "mitre_ttps": ["Remote Desktop Protocol", "Valid Accounts", "Exploit Public-Facing Application"],
        "targeted_sectors": ["Finance", "Banking"],
        "targeted_countries": ["Mexico"],
        "iocs": ["@sombra_red (Telegram)"],
        "threat_actor_profile": "Financially motivated broker, moderate sophistication, likely Mexico-based.",
        "analyst_actions": ["Block RDP exposure on perimeter firewalls", "Alert financial sector ISACs", "Monitor Telegram handle"],
        "intelligence_gaps": ["Specific institution not identified"],
        "confidence": "HIGH",
        "tlp": "TLP:RED",
    },
    {
        "severity": "HIGH",
        "threat_type": "Hacktivist Data Extortion",
        "summary": "A hacktivist group threatening to leak 2.3 million Argentine taxpayer records unless the Ministry of Economy complies with transparency demands. Friday deadline creates urgency.",
        "mitre_ttps": ["Data Exfiltration", "Defacement"],
        "targeted_sectors": ["Government"],
        "targeted_countries": ["Argentina"],
        "iocs": ["#OpArgentina"],
        "threat_actor_profile": "Ideologically motivated hacktivist, Anonymous affiliation claimed.",
        "analyst_actions": ["Notify CERT.ar", "Assess Ministry network for compromise", "Prepare communications plan"],
        "intelligence_gaps": ["Data possession unverified"],
        "confidence": "MEDIUM",
        "tlp": "TLP:AMBER",
    },
    {
        "severity": "CRITICAL",
        "threat_type": "Ransomware-as-a-Service",
        "summary": "New RaaS group 'Cóndor Negro' recruiting affiliates and claims to have compromised three Santiago healthcare facilities. Above-market 75% payout suggests aggressive expansion.",
        "mitre_ttps": ["Data Encrypted for Impact", "Inhibit System Recovery"],
        "targeted_sectors": ["Healthcare"],
        "targeted_countries": ["Chile"],
        "iocs": ["Cóndor Negro (group name)"],
        "threat_actor_profile": "Sophisticated financially motivated RaaS operator with strong OPSEC.",
        "analyst_actions": ["Alert Chilean healthcare CISOs", "Share TTPs with CSIRT Chile", "Hunt for I2P C2 beaconing"],
        "intelligence_gaps": ["Ransomware strain not yet identified"],
        "confidence": "HIGH",
        "tlp": "TLP:RED",
    },
]


def run_centinela_engine(use_mock=True):
    """Run Centinela analysis on Spanish-language threat feeds."""
    results = []
    for i, feed in enumerate(CENTINELA_FEEDS):
        if use_mock:
            intel = CENTINELA_MOCK[i].copy()
        else:
            prompt = f"""You are a senior threat intelligence analyst specializing in Spanish-language cybercriminal ecosystems.
Analyze this post and return ONLY a valid JSON object:
{{
  "severity": "CRITICAL|HIGH|MEDIUM|LOW",
  "threat_type": "concise threat category",
  "summary": "2-3 sentence analytical summary in English",
  "mitre_ttps": ["MITRE ATT&CK technique names"],
  "targeted_sectors": ["sectors"],
  "targeted_countries": ["countries"],
  "iocs": ["indicators"],
  "threat_actor_profile": "brief profile",
  "analyst_actions": ["2-3 actions"],
  "intelligence_gaps": ["gaps"],
  "confidence": "HIGH|MEDIUM|LOW",
  "tlp": "TLP:RED|TLP:AMBER|TLP:GREEN"
}}

POST: {feed['raw_text']}
SOURCE: {feed['source']} | REGION: {feed['region']}"""
            message = client.messages.create(
                model="claude-opus-4-5",
                max_tokens=1024,
                messages=[{"role": "user", "content": prompt}],
            )
            raw = message.content[0].text.strip().replace("```json", "").replace("```", "")
            intel = json.loads(raw)

        intel.update({
            "feed_id": feed["id"],
            "source": feed["source"],
            "region": feed["region"],
            "timestamp": feed["timestamp"],
            "raw_text": feed["raw_text"],
            "module": "centinela",
        })
        results.append(intel)

        # Update session stats
        sev = intel.get("severity", "LOW")
        SESSION["severity_counts"][sev] = SESSION["severity_counts"].get(sev, 0) + 1
        SESSION["total_analyzed"] += 1

    SESSION["centinela_reports"] = results
    return results

In [7]:
SENTINEL_IOCS = [
    "185.220.101.45",
    "paypa1-secure-login.com",
    "44d88612fea8a8f36de82e1278abb02f",
    """From: security@paypa1-secure.com
Subject: URGENT: Your account has been limited
Dear Customer, your PayPal account has been limited.
Click here: http://paypa1-secure.com/login
Failure to act in 24 hours will result in suspension.""",
]

SENTINEL_MOCK = [
    {
        "verdict": "MALICIOUS", "severity": "HIGH", "confidence": "HIGH",
        "threat_type": "C2 Server / Tor Exit Node",
        "summary": "This IP is a known Tor exit node with significant malicious activity reported. It has been flagged by 12 VirusTotal engines and shows an AbuseIPDB confidence score of 87%, associated with scanning and brute force activity.",
        "key_findings": ["12/80 VirusTotal engines flagged as malicious", "AbuseIPDB score: 87%", "143 abuse reports in 90 days", "Hosted by suspicious data center in Russia"],
        "recommended_actions": ["Block at perimeter firewall immediately", "Search SIEM for internal hosts communicating with this IP", "Add to threat intel watchlist"],
        "false_positive_likelihood": "LOW",
        "mitre_ttps": ["Command and Control", "Proxy", "Ingress Tool Transfer"],
        "ioc_type": "ip",
    },
    {
        "verdict": "MALICIOUS", "severity": "HIGH", "confidence": "HIGH",
        "threat_type": "Phishing Infrastructure / Typosquatting",
        "summary": "This domain is a typosquatted PayPal impersonation site registered recently with no legitimate business purpose. Multiple VirusTotal engines have flagged it as phishing infrastructure.",
        "key_findings": ["8/55 VirusTotal engines flagged as phishing", "Domain registered within last 7 days", "Typosquats 'paypal.com' using '1' instead of 'l'", "No HTTPS — uses plain HTTP"],
        "recommended_actions": ["Block at DNS and proxy layer", "Report to PayPal abuse team", "Search email logs for links to this domain"],
        "false_positive_likelihood": "LOW",
        "mitre_ttps": ["Phishing", "Spearphishing Link"],
        "ioc_type": "domain",
    },
    {
        "verdict": "MALICIOUS", "severity": "CRITICAL", "confidence": "HIGH",
        "threat_type": "Known Malware Sample",
        "summary": "This MD5 hash matches a known malware sample flagged by 52 of 65 VirusTotal engines. The sample is associated with credential stealing and data exfiltration activity.",
        "key_findings": ["52/65 VirusTotal engines flagged as malicious", "Associated with credential stealing malware family", "No harmless detections — high confidence", "Matches known threat actor toolset"],
        "recommended_actions": ["Isolate any host where this file was found", "Conduct full forensic investigation", "Submit to sandbox for dynamic analysis"],
        "false_positive_likelihood": "LOW",
        "mitre_ttps": ["Credential Dumping", "Exfiltration Over C2 Channel"],
        "ioc_type": "hash_md5",
    },
    {
        "verdict": "MALICIOUS", "severity": "HIGH", "confidence": "HIGH",
        "threat_type": "Phishing Email — Brand Impersonation",
        "summary": "This email is a PayPal phishing attempt using a typosquatted sender domain and urgency tactics to steal user credentials. The domain was registered recently and has no affiliation with PayPal.",
        "key_findings": ["Sender domain 'paypa1-secure.com' is not affiliated with PayPal", "Urgency language designed to bypass critical thinking", "Link directs to HTTP (not HTTPS) phishing page", "Generic greeting — not personalized"],
        "recommended_actions": ["Block sender domain at email gateway", "Alert users who may have received similar emails", "Submit URL to Google Safe Browsing"],
        "false_positive_likelihood": "LOW",
        "mitre_ttps": ["Phishing", "User Execution", "Spearphishing Link"],
        "ioc_type": "email",
    },
]

VT_MOCK = [
    {"malicious": 12, "suspicious": 3, "harmless": 45, "undetected": 20, "total": 80},
    {"malicious": 8, "suspicious": 2, "harmless": 30, "undetected": 15, "total": 55},
    {"malicious": 52, "suspicious": 5, "harmless": 0, "undetected": 8, "total": 65},
    None,
]

ABUSE_MOCK = {
    "abuse_confidence_score": 87, "total_reports": 143,
    "country": "RU", "isp": "Hosting Provider LLC",
    "domain": "suspicious-host.ru", "is_tor": True,
    "is_whitelisted": False, "usage_type": "Data Center/Web Hosting/Transit",
}


def detect_ioc_type(ioc):
    ioc = ioc.strip()
    if re.match(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$", ioc):
        return "ip"
    if re.match(r"^[a-fA-F0-9]{32}$", ioc):
        return "hash_md5"
    if re.match(r"^[a-fA-F0-9]{40}$", ioc):
        return "hash_sha1"
    if re.match(r"^[a-fA-F0-9]{64}$", ioc):
        return "hash_sha256"
    if re.match(r"^[^@]+@[^@]+\.[^@]+$", ioc):
        return "email"
    if re.match(r"^([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}$", ioc):
        return "domain"
    if "\n" in ioc or "Subject:" in ioc or "From:" in ioc:
        return "email"
    return "unknown"


def run_sentinel_engine(use_mock=True):
    """Run Sentinel IOC analysis."""
    results = []
    for i, ioc in enumerate(SENTINEL_IOCS):
        ioc_type = detect_ioc_type(ioc)
        vt_data = VT_MOCK[i] if use_mock else None
        abuse_data = ABUSE_MOCK if (use_mock and ioc_type == "ip") else None

        if use_mock:
            analysis = SENTINEL_MOCK[i].copy()
        else:
            prompt = f"""You are a senior SOC analyst. Analyze this IOC and return ONLY valid JSON:
{{
  "verdict": "MALICIOUS|SUSPICIOUS|CLEAN|UNKNOWN",
  "severity": "CRITICAL|HIGH|MEDIUM|LOW",
  "confidence": "HIGH|MEDIUM|LOW",
  "threat_type": "concise category",
  "summary": "2-3 sentence verdict",
  "key_findings": ["3-4 findings"],
  "recommended_actions": ["2-3 actions"],
  "false_positive_likelihood": "HIGH|MEDIUM|LOW",
  "mitre_ttps": ["MITRE technique names"]
}}
IOC: {ioc[:500]}
TYPE: {ioc_type}
VIRUSTOTAL: {json.dumps(vt_data)}"""
            message = client.messages.create(
                model="claude-opus-4-5",
                max_tokens=1024,
                messages=[{"role": "user", "content": prompt}],
            )
            raw = message.content[0].text.strip().replace("```json", "").replace("```", "")
            analysis = json.loads(raw)

        result = {
            "ioc": ioc[:80] + "..." if len(ioc) > 80 else ioc,
            "ioc_type": ioc_type,
            "analysis": analysis,
            "vt_data": vt_data,
            "abuse_data": abuse_data,
            "module": "sentinel",
        }
        results.append(result)

        verdict = analysis.get("verdict", "UNKNOWN")
        sev = analysis.get("severity", "LOW")
        SESSION["verdict_counts"][verdict] = SESSION["verdict_counts"].get(verdict, 0) + 1
        SESSION["severity_counts"][sev] = SESSION["severity_counts"].get(sev, 0) + 1
        SESSION["total_analyzed"] += 1

    SESSION["sentinel_reports"] = results
    return results


In [8]:
SEVERITY_COLORS = {
    "CRITICAL": ("#ff2d55", "#2a0a10"),
    "HIGH":     ("#ff9f0a", "#2a1a00"),
    "MEDIUM":   ("#ffd60a", "#2a2200"),
    "LOW":      ("#30d158", "#0a1f10"),
}

VERDICT_COLORS = {
    "MALICIOUS":  ("#ff2d55", "#2a0a10"),
    "SUSPICIOUS": ("#ff9f0a", "#2a1a00"),
    "CLEAN":      ("#30d158", "#0a1f10"),
    "UNKNOWN":    ("#636366", "#1a1a1a"),
}

TLP_COLORS = {
    "TLP:RED": "#ff2d55", "TLP:AMBER": "#ff9f0a", "TLP:GREEN": "#30d158",
}


def pill(text, color="#aaa", bg="rgba(255,255,255,0.08)"):
    return (
        f'<span style="display:inline-block;padding:2px 10px;border-radius:20px;'
        f'font-size:11px;font-weight:600;background:{bg};color:{color};'
        f'border:1px solid {color}40;margin:2px;">{text}</span>'
    )


def section_html(title, items):
    if not items:
        return ""
    if isinstance(items, list):
        content = "".join(f"<li style='margin:3px 0;color:#ccc;font-size:13px;'>{i}</li>" for i in items)
        content = f"<ul style='margin:6px 0 0 16px;padding:0;'>{content}</ul>"
    else:
        content = f"<p style='margin:6px 0 0;color:#ccc;font-size:13px;line-height:1.5;'>{items}</p>"
    return (
        f'<div style="margin-top:14px;">'
        f'<div style="font-size:10px;letter-spacing:0.12em;color:#666;font-weight:700;'
        f'text-transform:uppercase;margin-bottom:2px;">{title}</div>{content}</div>'
    )


def render_watchtower_header():
    s = SESSION
    critical = s["severity_counts"].get("CRITICAL", 0)
    high = s["severity_counts"].get("HIGH", 0)
    malicious = s["verdict_counts"].get("MALICIOUS", 0)

    return f"""
    <div style="font-family:'Courier New',monospace;background:#050505;
                border:1px solid #1a1a1a;border-radius:8px;padding:28px;margin-bottom:16px;">
      <div style="color:#bf5fff;font-size:26px;font-weight:700;letter-spacing:0.15em;
                  text-shadow:0 0 20px rgba(191,95,255,0.5);">◈ WATCHTOWER</div>
      <div style="color:#444;font-size:11px;letter-spacing:0.3em;margin-top:4px;">
        UNIFIED THREAT INTELLIGENCE PLATFORM · CENTINELA + SENTINEL
      </div>
      <div style="border-top:1px solid #1a1a1a;margin:20px 0;"></div>
      <div style="display:flex;gap:32px;flex-wrap:wrap;">
        <div style="text-align:center;">
          <div style="font-size:32px;font-weight:700;color:#eee;">{s['total_analyzed']}</div>
          <div style="font-size:10px;color:#555;letter-spacing:0.1em;">TOTAL ANALYZED</div>
        </div>
        <div style="text-align:center;">
          <div style="font-size:32px;font-weight:700;color:#ff2d55;">{critical}</div>
          <div style="font-size:10px;color:#555;letter-spacing:0.1em;">CRITICAL</div>
        </div>
        <div style="text-align:center;">
          <div style="font-size:32px;font-weight:700;color:#ff9f0a;">{high}</div>
          <div style="font-size:10px;color:#555;letter-spacing:0.1em;">HIGH</div>
        </div>
        <div style="text-align:center;">
          <div style="font-size:32px;font-weight:700;color:#ff2d55;">{malicious}</div>
          <div style="font-size:10px;color:#555;letter-spacing:0.1em;">MALICIOUS</div>
        </div>
        <div style="text-align:center;">
          <div style="font-size:32px;font-weight:700;color:#bf5fff;">
            {len(s['centinela_reports'])}
          </div>
          <div style="font-size:10px;color:#555;letter-spacing:0.1em;">THREAT FEEDS</div>
        </div>
        <div style="text-align:center;">
          <div style="font-size:32px;font-weight:700;color:#5fb3ff;">
            {len(s['sentinel_reports'])}
          </div>
          <div style="font-size:10px;color:#555;letter-spacing:0.1em;">IOCs TRIAGED</div>
        </div>
      </div>
      <div style="margin-top:16px;font-size:11px;color:#333;">
        Session started: {s['started']}
      </div>
    </div>
    """


def render_section_header(title, subtitle, color):
    return f"""
    <div style="font-family:'Courier New',monospace;background:#0a0a0a;
                border-left:4px solid {color};border-radius:4px;
                padding:14px 20px;margin:20px 0 8px;">
      <div style="font-size:14px;font-weight:700;color:{color};letter-spacing:0.1em;">{title}</div>
      <div style="font-size:11px;color:#444;margin-top:2px;">{subtitle}</div>
    </div>
    """


def render_centinela_card(intel):
    sev = intel.get("severity", "MEDIUM")
    sev_color, sev_bg = SEVERITY_COLORS.get(sev, ("#ffd60a", "#2a2200"))
    tlp = intel.get("tlp", "TLP:AMBER")
    tlp_color = TLP_COLORS.get(tlp, "#ff9f0a")
    ttp_pills = "".join(pill(t, "#bf5fff", "rgba(191,95,255,0.1)") for t in intel.get("mitre_ttps", []))
    sector_pills = "".join(pill(s, "#5fb3ff", "rgba(95,179,255,0.1)") for s in intel.get("targeted_sectors", []))

    return f"""
    <div style="font-family:'Courier New',monospace;background:#0d0d0d;
                border:1px solid {sev_color}40;border-left:4px solid {sev_color};
                border-radius:8px;padding:20px;margin:8px 0;
                box-shadow:0 4px 24px rgba(0,0,0,0.5);">
      <div style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
        <div>
          <div style="font-size:11px;color:#555;">{intel['feed_id']} · {intel['source']} · {intel['timestamp']}</div>
          <div style="font-size:15px;font-weight:700;color:#eee;margin-top:4px;">{intel.get('threat_type','')}</div>
          <div style="font-size:12px;color:#888;">📍 {intel['region']}</div>
        </div>
        <div style="display:flex;flex-direction:column;align-items:flex-end;gap:4px;">
          <span style="padding:4px 12px;border-radius:4px;font-size:12px;font-weight:800;
                       background:{sev_bg};color:{sev_color};border:1px solid {sev_color};">⚠ {sev}</span>
          <span style="font-size:11px;font-weight:700;color:{tlp_color};">{tlp}</span>
          <span style="font-size:11px;color:#555;">Confidence: <span style="color:#aaa;">{intel.get('confidence','?')}</span></span>
        </div>
      </div>
      <div style="border-top:1px solid #1a1a1a;margin:12px 0;"></div>
      <div style="background:#111;border-left:3px solid #333;padding:8px 12px;
                  border-radius:4px;font-size:12px;color:#666;font-style:italic;">
        "{intel['raw_text'][:200]}..."
      </div>
      {section_html("📋 Summary", intel.get('summary',''))}
      {('<div style="margin-top:12px;"><div style="font-size:10px;letter-spacing:0.12em;color:#666;font-weight:700;text-transform:uppercase;margin-bottom:4px;">🎯 Sectors</div>' + sector_pills + '</div>') if sector_pills else ''}
      {('<div style="margin-top:12px;"><div style="font-size:10px;letter-spacing:0.12em;color:#666;font-weight:700;text-transform:uppercase;margin-bottom:4px;">⚔ TTPs</div>' + ttp_pills + '</div>') if ttp_pills else ''}
      {section_html("✅ Analyst Actions", intel.get('analyst_actions', []))}
    </div>"""


def render_sentinel_card(result):
    analysis = result["analysis"]
    verdict = analysis.get("verdict", "UNKNOWN")
    verdict_color, verdict_bg = VERDICT_COLORS.get(verdict, ("#636366", "#1a1a1a"))
    sev = analysis.get("severity", "MEDIUM")
    sev_color = SEVERITY_COLORS.get(sev, ("#ffd60a", "#2a2200"))[0]
    vt = result.get("vt_data")
    abuse = result.get("abuse_data")
    ttp_pills = "".join(pill(t, "#5fb3ff", "rgba(95,179,255,0.1)") for t in analysis.get("mitre_ttps", []))

    vt_html = ""
    if vt:
        vt_html = f"""
        <div style="background:#111;border-radius:6px;padding:10px;margin-top:12px;">
          <div style="font-size:10px;color:#666;font-weight:700;letter-spacing:0.1em;
                      text-transform:uppercase;margin-bottom:8px;">🛡 VirusTotal</div>
          <div style="display:flex;gap:16px;">
            <div style="text-align:center;">
              <div style="font-size:18px;font-weight:700;color:#ff2d55;">{vt.get('malicious',0)}</div>
              <div style="font-size:10px;color:#555;">Malicious</div>
            </div>
            <div style="text-align:center;">
              <div style="font-size:18px;font-weight:700;color:#ff9f0a;">{vt.get('suspicious',0)}</div>
              <div style="font-size:10px;color:#555;">Suspicious</div>
            </div>
            <div style="text-align:center;">
              <div style="font-size:18px;font-weight:700;color:#eee;">{vt.get('total',0)}</div>
              <div style="font-size:10px;color:#555;">Total</div>
            </div>
          </div>
        </div>"""

    abuse_html = ""
    if abuse:
        score = abuse.get("abuse_confidence_score", 0)
        score_color = "#ff2d55" if score > 50 else "#ff9f0a" if score > 20 else "#30d158"
        abuse_html = f"""
        <div style="background:#111;border-radius:6px;padding:10px;margin-top:8px;">
          <div style="font-size:10px;color:#666;font-weight:700;letter-spacing:0.1em;
                      text-transform:uppercase;margin-bottom:8px;">🚨 AbuseIPDB</div>
          <div style="display:flex;gap:16px;align-items:center;">
            <div style="text-align:center;">
              <div style="font-size:18px;font-weight:700;color:{score_color};">{score}%</div>
              <div style="font-size:10px;color:#555;">Abuse Score</div>
            </div>
            <div style="font-size:12px;color:#888;line-height:1.8;">
              Reports: <span style="color:#ccc;">{abuse.get('total_reports',0)}</span> ·
              Country: <span style="color:#ccc;">{abuse.get('country','?')}</span> ·
              Tor: <span style="color:#ccc;">{'Yes' if abuse.get('is_tor') else 'No'}</span>
            </div>
          </div>
        </div>"""

    ioc_display = result['ioc'][:60] + "..." if len(result['ioc']) > 60 else result['ioc']

    return f"""
    <div style="font-family:'Courier New',monospace;background:#0d0d0d;
                border:1px solid {verdict_color}40;border-left:4px solid {verdict_color};
                border-radius:8px;padding:20px;margin:8px 0;
                box-shadow:0 4px 24px rgba(0,0,0,0.5);">
      <div style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
        <div>
          <div style="font-size:11px;color:#555;">{result['ioc_type'].upper()} · {datetime.now(UTC).strftime('%Y-%m-%d %H:%M UTC')}</div>
          <div style="font-size:15px;font-weight:700;color:#eee;margin-top:4px;word-break:break-all;">{ioc_display}</div>
          <div style="font-size:12px;color:#888;">{analysis.get('threat_type','')}</div>
        </div>
        <div style="display:flex;flex-direction:column;align-items:flex-end;gap:4px;">
          <span style="padding:4px 12px;border-radius:4px;font-size:12px;font-weight:800;
                       background:{verdict_bg};color:{verdict_color};border:1px solid {verdict_color};">● {verdict}</span>
          <span style="font-size:11px;color:{sev_color};font-weight:700;">⚠ {sev}</span>
          <span style="font-size:11px;color:#555;">Confidence: <span style="color:#aaa;">{analysis.get('confidence','?')}</span></span>
        </div>
      </div>
      <div style="border-top:1px solid #1a1a1a;margin:12px 0;"></div>
      {section_html("📋 Summary", analysis.get('summary',''))}
      {vt_html}
      {abuse_html}
      {section_html("🔍 Key Findings", analysis.get('key_findings', []))}
      {('<div style="margin-top:12px;"><div style="font-size:10px;letter-spacing:0.12em;color:#666;font-weight:700;text-transform:uppercase;margin-bottom:4px;">⚔ TTPs</div>' + ttp_pills + '</div>') if ttp_pills else ''}
      {section_html("✅ Recommended Actions", analysis.get('recommended_actions', []))}
    </div>"""


In [9]:
def run_watchtower(use_mock=True):
    """
    Run the full Watchtower unified dashboard.

    Args:
        use_mock: Set True to demo without API keys. Set False for live analysis.
    """
    print("=" * 60)
    print("  WATCHTOWER — Unified Threat Intelligence Platform")
    print("=" * 60)
    print(f"  Session: {SESSION['started']}")
    print(f"  Mode: {'DEMO (mock data)' if use_mock else 'LIVE (real APIs)'}")
    print("=" * 60)

    # Run Centinela
    print("\n[1/2] Running CENTINELA — Spanish-Language Threat Feeds...")
    centinela_results = run_centinela_engine(use_mock=use_mock)
    print(f"  ✓ Analyzed {len(centinela_results)} threat feed items")

    # Run Sentinel
    print("\n[2/2] Running SENTINEL — IOC Triage...")
    sentinel_results = run_sentinel_engine(use_mock=use_mock)
    print(f"  ✓ Triaged {len(sentinel_results)} indicators")

    print("\n[WATCHTOWER] Rendering unified dashboard...\n")

    # Render dashboard header
    display(HTML(render_watchtower_header()))

    # Render Centinela section
    display(HTML(render_section_header(
        "◈ CENTINELA — Spanish-Language Threat Intelligence",
        "Dark web forums · Telegram channels · Underground marketplaces",
        "#bf5fff"
    )))
    for intel in centinela_results:
        display(HTML(render_centinela_card(intel)))

    # Render Sentinel section
    display(HTML(render_section_header(
        "◈ SENTINEL — IOC Triage & Analysis",
        "IP addresses · Domains · File hashes · Phishing emails",
        "#5fb3ff"
    )))
    for result in sentinel_results:
        display(HTML(render_sentinel_card(result)))

    # Save full session report
    session_export = {
        "session": SESSION,
        "centinela_reports": centinela_results,
        "sentinel_reports": [
            {**r, "analysis": r["analysis"]} for r in sentinel_results
        ],
        "generated_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M UTC"),
    }

    with open("watchtower_report.json", "w", encoding="utf-8") as f:
        json.dump(session_export, f, ensure_ascii=False, indent=2, default=str)

    print(f"\n[WATCHTOWER] Dashboard complete.")
    print(f"  Total analyzed: {SESSION['total_analyzed']}")
    print(f"  Full report saved to watchtower_report.json")
    return session_export


In [10]:
if __name__ == "__main__":
    # Set use_mock=True to run without API keys (demo mode)
    # Set use_mock=False once API keys are configured in Colab Secrets
    report = run_watchtower(use_mock=True)

  WATCHTOWER — Unified Threat Intelligence Platform
  Session: 2026-04-28 19:33 UTC
  Mode: DEMO (mock data)

[1/2] Running CENTINELA — Spanish-Language Threat Feeds...
  ✓ Analyzed 3 threat feed items

[2/2] Running SENTINEL — IOC Triage...
  ✓ Triaged 4 indicators

[WATCHTOWER] Rendering unified dashboard...




[WATCHTOWER] Dashboard complete.
  Total analyzed: 7
  Full report saved to watchtower_report.json
